In [5]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pickle
import os

print("🚀 Step 1: Loading Processed Data & Graph...")

data_path = r"C:\Users\ANISH\Desktop\FRP\ecstgnn\ec-st-gnn-airpollution-forecasting\data\processed\processed_delhi_data.csv" 
nodes_path = r"C:\Users\ANISH\Desktop\FRP\ecstgnn\ec-st-gnn-airpollution-forecasting\data\processed\node_metadata.csv"

df = pd.read_csv(data_path)
nodes = pd.read_csv(nodes_path)

# 1. Map station names to integer IDs to ensure perfect matrix alignment
station_to_id = dict(zip(nodes['station'], nodes['station_id']))
df['station_id'] = df['station'].map(station_to_id)

# 2. Sort chronologically, then by station ID
df = df.sort_values(by=['datetime', 'station_id'])

# 3. Define Features (Target variable 'pm25' must be first!)
features = [
    'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'temperature', 
    'humidity', 'wind_speed', 'visibility', 'aqi', 
    'is_weekend', 'hour_sin', 'hour_cos'
]

num_nodes = len(nodes)
num_features = len(features)
num_timesteps = len(df['datetime'].unique())

print(f"Dataset bounds: {num_timesteps} Time Steps, {num_nodes} Nodes, {num_features} Features")

# 4. Reshape into 3D Numpy Array: [Time, Nodes, Features]
raw_data = df[features].values
reshaped_data = raw_data.reshape(num_timesteps, num_nodes, num_features)

print("🚀 Step 2: Scaling Data for Neural Network...")
scaler = StandardScaler()
# Flatten to 2D to scale, then reshape back to 3D
scaled_data_2d = scaler.fit_transform(reshaped_data.reshape(-1, num_features))
scaled_data = scaled_data_2d.reshape(num_timesteps, num_nodes, num_features)

# Save the scaler using absolute paths
scaler_dir = r"C:\Users\ANISH\Desktop\FRP\ecstgnn\ec-st-gnn-airpollution-forecasting\app\models"
os.makedirs(scaler_dir, exist_ok=True)
with open(os.path.join(scaler_dir, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
print("✅ Scaler saved to app/models/scaler.pkl")

print("🚀 Step 3: Initializing PyTorch Sliding Window Dataset...")
# --- THE CORE PYTORCH DATASET CLASS ---
class STGNNDataset(Dataset):
    def __init__(self, data, seq_len, pre_len):
        self.data = data
        self.seq_len = seq_len  # History (e.g., past 4 days)
        self.pre_len = pre_len  # Forecast (e.g., next 24 hours)
        
    def __len__(self):
        return len(self.data) - self.seq_len - self.pre_len + 1
        
    def __getitem__(self, index):
        # X: All features for the historical window
        X = self.data[index : index + self.seq_len]
        
        # y: ONLY PM2.5 (feature index 0) for the future window
        y = self.data[index + self.seq_len : index + self.seq_len + self.pre_len, :, 0]
        
        return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# Define Window Sizes: 16 steps back (4 days), 4 steps forward (1 day)
seq_len = 16 
pre_len = 4  

dataset = STGNNDataset(scaled_data, seq_len=seq_len, pre_len=pre_len)

# Create DataLoader to feed the GPU in batches of 32
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Test the loader!
for batch_x, batch_y in dataloader:
    print(f"\n✅ SUCCESS! Batch Input (X) Shape: {batch_x.shape}")
    print("    -> [Batch Size, Time History, Nodes, Features]")
    print(f"✅ SUCCESS! Batch Target (y) Shape: {batch_y.shape}")
    print("    -> [Batch Size, Forecast Horizon, Nodes]")
    break

🚀 Step 1: Loading Processed Data & Graph...
Dataset bounds: 8767 Time Steps, 23 Nodes, 14 Features
🚀 Step 2: Scaling Data for Neural Network...
✅ Scaler saved to app/models/scaler.pkl
🚀 Step 3: Initializing PyTorch Sliding Window Dataset...

✅ SUCCESS! Batch Input (X) Shape: torch.Size([32, 16, 23, 14])
    -> [Batch Size, Time History, Nodes, Features]
✅ SUCCESS! Batch Target (y) Shape: torch.Size([32, 4, 23])
    -> [Batch Size, Forecast Horizon, Nodes]


In [7]:
import torch.nn as nn
import torch.nn.functional as F

print("🚀 Step 4: Compiling ST-GNN Architecture...")

class CausalSTGNN(nn.Module):
    def __init__(self, num_nodes, num_features, hidden_dim, forecast_horizon):
        super(CausalSTGNN, self).__init__()
        self.num_nodes = num_nodes
        self.hidden_dim = hidden_dim
        
        # 1. Temporal Component (GRU)
        # Learns the time-series trends from the 14 features
        self.gru = nn.GRU(input_size=num_features, hidden_size=hidden_dim, batch_first=True)
        
        # 2. Spatial Component (Graph Convolution)
        # Learns how to weigh the information passed between neighboring stations
        self.spatial_linear = nn.Linear(hidden_dim, hidden_dim)
        
        # 3. Forecasting Component
        # Maps the final spatio-temporal understanding to the future PM2.5 values
        self.predictor = nn.Linear(hidden_dim, forecast_horizon)
        
    def forward(self, x, adj):
        # x shape: [Batch, Time, Nodes, Features] -> [32, 16, 23, 14]
        batch_size, seq_len, num_nodes, num_features = x.shape
        
        # --- A. TEMPORAL PROCESSING ---
        # Reshape so the GRU can process all 23 nodes in parallel
        # New shape: [Batch * Nodes, Time, Features] -> [736, 16, 14]
        x_reshaped = x.transpose(1, 2).reshape(batch_size * num_nodes, seq_len, num_features)
        
        # Pass through GRU
        gru_out, hidden = self.gru(x_reshaped) 
        
        # Extract the final hidden state (the GRU's summary of the last 4 days)
        # Reshape back to separate the nodes: [Batch, Nodes, Hidden_Dim]
        last_hidden = hidden[-1].view(batch_size, num_nodes, self.hidden_dim)
        
        # --- B. SPATIAL PROCESSING (MESSAGE PASSING) ---
        # Multiply our historical summary by the spatial Adjacency Matrix
        # This is where wind paths and geographical proximity transfer pollution data!
        spatial_out = torch.matmul(adj, last_hidden) 
        spatial_out = F.relu(self.spatial_linear(spatial_out))
        
        # --- C. PREDICTION ---
        # Output shape: [Batch, Nodes, Forecast_Horizon]
        predictions = self.predictor(spatial_out)
        
        # Transpose to match our Target (y) shape exactly: [Batch, Forecast_Horizon, Nodes]
        return predictions.transpose(1, 2)

# Initialize the Model
# 14 Features in, 64-neuron hidden "brain", 4 steps (24hrs) out
model = CausalSTGNN(num_nodes=num_nodes, num_features=num_features, hidden_dim=64, forecast_horizon=pre_len)

# Load the Adjacency Matrix we built yesterday!
# Updated Absolute Path
adj_matrix_path = r"C:\Users\ANISH\Desktop\FRP\ecstgnn\ec-st-gnn-airpollution-forecasting\data\processed\adjacency_matrix.npy"
# Convert numpy array to PyTorch float tensor
adj_tensor = torch.tensor(np.load(adj_matrix_path), dtype=torch.float32)

print("\n✅ Model initialized successfully!")
print(model)

# --- THE ULTIMATE SHAPE TEST ---
print("\n🚀 Step 5: Running a Forward Pass Test...")
# Grab one batch from our dataloader
sample_x, sample_y = next(iter(dataloader))

# Push it through the untrained model
with torch.no_grad(): # No training yet, just testing the pipes
    sample_prediction = model(sample_x, adj_tensor)

print(f"Target (y) Shape:      {sample_y.shape}")
print(f"Prediction (ŷ) Shape:  {sample_prediction.shape}")

if sample_y.shape == sample_prediction.shape:
    print("\n🎉 SUCCESS! The network architecture is mathematically sound. Ready for training loop.")
else:
    print("\n❌ ERROR: Shape mismatch.")

🚀 Step 4: Compiling ST-GNN Architecture...

✅ Model initialized successfully!
CausalSTGNN(
  (gru): GRU(14, 64, batch_first=True)
  (spatial_linear): Linear(in_features=64, out_features=64, bias=True)
  (predictor): Linear(in_features=64, out_features=4, bias=True)
)

🚀 Step 5: Running a Forward Pass Test...
Target (y) Shape:      torch.Size([32, 4, 23])
Prediction (ŷ) Shape:  torch.Size([32, 4, 23])

🎉 SUCCESS! The network architecture is mathematically sound. Ready for training loop.


In [8]:
import torch.optim as optim

print("🚀 Step 6: Initializing Training Engine...")

# 1. Define how we measure "wrongness" (Mean Squared Error)
criterion = nn.MSELoss()

# 2. Define the Optimizer (Adam is the industry standard for GNNs)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Set the number of epochs (times the model sees the entire dataset)
epochs = 10

print("🔥 Starting Training Loop...")
model.train() # Tell PyTorch we are in training mode

for epoch in range(epochs):
    total_loss = 0
    
    # Loop through our flashcards batch by batch
    for batch_x, batch_y in dataloader:
        
        # Zero the gradients (clear the memory of the last step)
        optimizer.zero_grad()
        
        # Forward Pass: Ask the model to make a prediction
        predictions = model(batch_x, adj_tensor)
        
        # Calculate the Loss: How far off was the prediction from the truth?
        loss = criterion(predictions, batch_y)
        
        # Backward Pass: Calculate the mathematical adjustments needed
        loss.backward()
        
        # Optimize: Actually update the brain's weights
        optimizer.step()
        
        total_loss += loss.item()
        
    # Calculate average loss for this epoch
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{epochs}] | Average Loss: {avg_loss:.4f}")

print("\n✅ Training Complete!")

print("🚀 Step 7: Saving the Trained Model...")
# Use absolute paths so it drops perfectly into your frontend's model folder
model_dir = r"C:\Users\ANISH\Desktop\FRP\ecstgnn\ec-st-gnn-airpollution-forecasting\app\models"
os.makedirs(model_dir, exist_ok=True)
model_save_path = os.path.join(model_dir, "stgnn_weights.pth")

# Save just the mathematical weights (keeps the file size very small)
torch.save(model.state_dict(), model_save_path)

print(f"💾 Model weights successfully saved to: {model_save_path}")
print("🎯 PHASE 2 IS OFFICIALLY COMPLETE!")

🚀 Step 6: Initializing Training Engine...
🔥 Starting Training Loop...
Epoch [1/10] | Average Loss: 0.0814
Epoch [2/10] | Average Loss: 0.0460
Epoch [3/10] | Average Loss: 0.0442
Epoch [4/10] | Average Loss: 0.0437
Epoch [5/10] | Average Loss: 0.0419
Epoch [6/10] | Average Loss: 0.0416
Epoch [7/10] | Average Loss: 0.0406
Epoch [8/10] | Average Loss: 0.0414
Epoch [9/10] | Average Loss: 0.0413
Epoch [10/10] | Average Loss: 0.0397

✅ Training Complete!
🚀 Step 7: Saving the Trained Model...
💾 Model weights successfully saved to: C:\Users\ANISH\Desktop\FRP\ecstgnn\ec-st-gnn-airpollution-forecasting\app\models\stgnn_weights.pth
🎯 PHASE 2 IS OFFICIALLY COMPLETE!
